BUILD DATASET (gold standard in one go)

In [1]:
import os
import amsterdamumcdb as adb
import pandas as pd
from google.cloud import bigquery

#login terminal commadns
#brew install --cask google-cloud-sdk   # skip if already installedd
#gcloud auth application-default login
#gcloud auth application-default set-quota-project liquid-folio-513114-d9

projectId = "liquid-folio-513114-d9"
datasetId = "amsterdamumcdb.version1_5_0"
location = "eu"
searchPattern = "extub|detub|reintub|intub"

os.environ["GOOGLE_CLOUD_PROJECT"] = projectId
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 200)

#default dataset
#fail above 20gb to be sure and not burn everythin
jobConfig = bigquery.QueryJobConfig(default_dataset=datasetId, maximum_bytes_billed=20*1024**3)
client = bigquery.Client(project=projectId, location=location, default_query_job_config=jobConfig)

#Found ventilation mode in measurements, it shows what mode of ventilation is done
ventilationModeConceptId = 3004921

#values of ventilatie mode (set) - hope this si correct but maybe ask prof? Or in PP
invasiveModes = ["PS/CPAP", "PC", "PC (No trig)", "PS/CPAP (trig)", "VC", "NAVA", "SIMV(PC)+PS", "SIMV(VC)+PS", "Bi Vente", "VC (No trig)", "VC (trig)", "PRVC (No trig)", "PRVC", "VS", "PRVC (trig)"]

#admissions with oxygen through a tracheostoma or a speaking cannula/valve <- maybe exclude as easier lets say in PP very clear so prof can see if correct
tracheostomySql = """
SELECT DISTINCT visit_occurrence_id FROM observation
WHERE provider_id IS NOT NULL AND observation_source_value = 'Toedieningsweg' AND TRIM(value_source_value) IN ('Trach.stoma', 'Spreekcanule', 'Spreekklepje')
"""

#NEW current group invasive ventilation no tracheostomy no cardiac surgery no planned surgery 
# (see above table cardiac mostly planned others have few planned so a bit of a bummer but cant delete those)
ventilatedSql = f"""
SELECT DISTINCT visit_occurrence_id, person_id
FROM measurement
WHERE provider_id IS NOT NULL AND measurement_concept_id = {ventilationModeConceptId} AND measurement_source_value = 'Ventilatie Mode (Set)'
AND TRIM(value_source_value) IN {tuple(invasiveModes)} AND visit_occurrence_id NOT IN (SELECT visit_occurrence_id FROM ({tracheostomySql}) WHERE visit_occurrence_id IS NOT NULL)
AND visit_occurrence_id NOT IN (
  SELECT visit_occurrence_id FROM visit_occurrence
  WHERE provider_id = 1 -- cardiochirurgie
  )
AND visit_occurrence_id NOT IN (
  SELECT visit_occurrence_id FROM observation
  WHERE provider_id IS NOT NULL
  AND observation_concept_id = 37154585
  AND observation_source_value = 'NICE Opname type'
  AND TRIM(value_source_value) = 'Geplande chirurgie'
  AND visit_occurrence_id IS NOT NULL
)
"""

#all invasive ventilation records of our group, sorted per patient in time mostly used later maybe pp
invasiveRecordsSql = f"""
WITH ventilated AS ({ventilatedSql})
SELECT measurement.person_id, measurement.visit_occurrence_id, measurement.measurement_datetime,
  TRIM(measurement.value_source_value) AS mode
FROM measurement
JOIN ventilated ON ventilated.visit_occurrence_id = measurement.visit_occurrence_id
WHERE measurement.provider_id IS NOT NULL AND measurement.measurement_concept_id = {ventilationModeConceptId} AND measurement.measurement_source_value = 'Ventilatie Mode (Set)'
AND TRIM(measurement.value_source_value) IN {tuple(invasiveModes)}
ORDER BY measurement.person_id, measurement.measurement_datetime
"""
invasiveRecords = client.query(invasiveRecordsSql).to_dataframe()

#oxygen route records of our group mostly used later and more often maybe pp
oxygenRecordsSql = f"""
WITH ventilated AS ({ventilatedSql})
SELECT observation.person_id, observation.observation_datetime, TRIM(observation.value_source_value) AS route
FROM observation
JOIN ventilated ON ventilated.visit_occurrence_id = observation.visit_occurrence_id
WHERE observation.provider_id IS NOT NULL
AND observation.observation_source_value = 'Toedieningsweg'
"""
oxygenRecords = client.query(oxygenRecordsSql).to_dataframe()

#death per patient in our group
deathsSql = f"""
WITH ventilated AS ({ventilatedSql})
SELECT DISTINCT death.person_id, death.death_datetime
FROM death
JOIN ventilated ON ventilated.person_id = death.person_id
"""
patientDeaths = client.query(deathsSql).to_dataframe()

#beleid records with time, to know the policy at each extubation
policyRecordsSql = f"""
WITH ventilated AS ({ventilatedSql})
SELECT observation.person_id, observation.observation_datetime AS policyTime, TRIM(observation.value_source_value) AS policy
FROM observation
JOIN ventilated ON ventilated.visit_occurrence_id = observation.visit_occurrence_id
WHERE observation.provider_id IS NOT NULL AND observation.observation_concept_id = 1761496 AND observation.observation_source_value = 'Beleid'
"""
policyRecords = client.query(policyRecordsSql).to_dataframe()

#ventilator and oxygen records together in one timeline per patient
ventilatorRows = invasiveRecords[["person_id", "measurement_datetime", "mode"]]
ventilatorRows.columns = ["person_id", "time", "value"]
ventilatorRows = ventilatorRows.assign(source="ventilator")
oxygenRows = oxygenRecords[["person_id", "observation_datetime", "route"]]
oxygenRows.columns = ["person_id", "time", "value"]
oxygenRows = oxygenRows.assign(source="oxygen")
allTimelines = pd.concat([ventilatorRows, oxygenRows]).sort_values(["person_id", "time"])

#tube ventilator or oxygen through the tube (kunstneus/ waterset/ dl-tube) and noTube = oxygen without a tube
allTimelines["state"] = "noTube"
allTimelines.loc[allTimelines["source"] == "ventilator", "state"] = "tube"
allTimelines.loc[allTimelines["value"].isin(["Kunstneus", "Waterset", "DL-tube"]), "state"] = "tube"

stateSwitches = []
previousRow = None
for row in allTimelines.itertuples():
    if previousRow is not None and row.person_id == previousRow.person_id and row.state != previousRow.state:
        stateSwitches.append({
            "person_id": row.person_id,
            "direction": previousRow.state + " -> " + row.state,
            "lastBefore": previousRow.time,
            "firstAfter": row.time,
            "valueBefore": previousRow.value,
            "valueAfter": row.value,
        })
    previousRow = row
stateSwitches = pd.DataFrame(stateSwitches)
stateSwitches["hoursBetween"] = (stateSwitches["firstAfter"] - stateSwitches["lastBefore"]).dt.total_seconds() / 3600

#per extubation the time until the patient is back on a tube (reintubation), if ever
#!!!!Within 1 hour reintubation real? maybe ask?
extubationList = []
for row in stateSwitches.itertuples():
    if row.direction == "tube -> noTube":
        extubationList.append({
            "person_id": row.person_id,
            "extubationTime": row.lastBefore,
            "valueBefore": row.valueBefore,
            "reintubationTime": None,
        })
    #a switch back to tube belongs to the last extubation of the same patient
    elif row.direction == "noTube -> tube" and len(extubationList) > 0 and extubationList[-1]["person_id"] == row.person_id:
        extubationList[-1]["reintubationTime"] = row.firstAfter
extubations = pd.DataFrame(extubationList)
extubations["reintubationTime"] = pd.to_datetime(extubations["reintubationTime"], utc=True)
extubations["hoursToReintubation"] = (extubations["reintubationTime"] - extubations["extubationTime"]).dt.total_seconds() / 3600

#time from extubation to death
extubationOutcomes = extubations.merge(patientDeaths, on="person_id", how="left")
extubationOutcomes["hoursToDeath"] = (extubationOutcomes["death_datetime"] - extubationOutcomes["extubationTime"]).dt.total_seconds() / 3600

#policy at moment of extubation is last beleid record before
policyRecords["person_id"] = policyRecords["person_id"].astype("int64")
extubationOutcomes = extubationOutcomes.sort_values("extubationTime")
policyRecords = policyRecords.sort_values("policyTime")
extubationOutcomes = pd.merge_asof(extubationOutcomes, policyRecords, left_on="extubationTime", right_on="policyTime", by="person_id", direction="backward")
extubationOutcomes["policy"] = extubationOutcomes["policy"].fillna("none")

#Below table shows sadly we cant say if died without reintubation that it was paliative as group 3 
# does not have biggest share. (Also small groups as a lot has been fixed already due to above criteria)

#comfort care at extubation leave out
goldStandard = extubationOutcomes[extubationOutcomes["policy"] != "III"].copy()

#failure defined as back on a tube or death within 7 days. 48h version as one paper does that maybe to look at sensitive cases more?
goldStandard["failure7Days"] = (goldStandard["hoursToReintubation"] <= 168) | (goldStandard["hoursToDeath"] <= 168)
goldStandard["failure48Hours"] = (goldStandard["hoursToReintubation"] <= 48) | (goldStandard["hoursToDeath"] <= 48)

#admission of each extubation is admission of the last invasive record before it
invasiveRecords["person_id"] = invasiveRecords["person_id"].astype("int64")
recordAdmissions = invasiveRecords[["person_id", "measurement_datetime", "visit_occurrence_id"]].sort_values("measurement_datetime")
goldStandard = goldStandard.sort_values("extubationTime")
goldStandard = pd.merge_asof(goldStandard, recordAdmissions, left_on="extubationTime", right_on="measurement_datetime", by="person_id", direction="backward")
goldStandard = goldStandard.drop(columns="measurement_datetime")

#attempt number within the admission 1 = first extubation etc I removed ones above 5 lets name it in PP were few records and seems unrealistic (or pat with tracheactomy as is easy to remove)
goldStandard = goldStandard.sort_values(["visit_occurrence_id", "extubationTime"])
goldStandard["attemptNumber"] = goldStandard.groupby("visit_occurrence_id").cumcount() + 1

#patients with 5 or more extubations one admission left out notd realistic weaning course I think PP
goldStandard = goldStandard[~goldStandard["person_id"].isin(
    goldStandard[goldStandard["attemptNumber"] >= 5]["person_id"])]

print(f"extubations: {len(goldStandard)}, patients: {goldStandard['person_id'].nunique()}")

print(goldStandard["attemptNumber"].value_counts().sort_index())
print(goldStandard.groupby("attemptNumber")["failure7Days"].mean())

extubations: 5369, patients: 4443
attemptNumber
1.0    4760
2.0     508
3.0      84
4.0      12
Name: count, dtype: int64
attemptNumber
1.0    0.186345
2.0    0.244094
3.0    0.273810
4.0    0.250000
Name: failure7Days, dtype: float64
